# Balancing and Cleaning the Dataset

This notebook is for loading our raw reviews dataset (`raw_reviews.csv`) and making it balanced so that our model trains properly. If we don't balance it, the model might get biased towards positive reviews since there are too many positive ones.

Steps we are doing here:
1. Load the raw reviews CSV file.
2. Rename/map columns like Rate to rating, Summary to review_text, and product_name to product_title.
3. Clean up null values and make sentiment labels consistent (Positive, Neutral, Negative).
4. Group reviews into categories like Electronics, Beauty, Home & Kitchen using keywords from product titles.
5. Balance the dataset by taking an equal number of reviews for Positive, Neutral, and Negative sentiments. We will sample 8800 reviews for each class because Neutral is the minority class.
6. Save the final balanced data as `balanced_reviews.csv` so we can use it for training.

In [ ]:
import pandas as pd
import numpy as np
import os

RAW_PATH = "../data/raw_reviews.csv"
BALANCED_PATH = "../data/balanced_reviews.csv"

if not os.path.exists(RAW_PATH):
    raise FileNotFoundError(f"Raw dataset not found at {RAW_PATH}")

print("Loading raw reviews...")
df = pd.read_csv(RAW_PATH)
print(f"Raw Shape: {df.shape}")
print(f"Original Columns: {df.columns.tolist()}")

In [ ]:
print("Checking for missing values in core columns...")
print(df[['Summary', 'Sentiment', 'Rate', 'product_name']].isnull().sum())

# Drop rows where core columns (text, label, product) are null
df_clean = df.dropna(subset=['Summary', 'Sentiment', 'product_name']).copy()
print(f"\nShape after dropping nulls: {df_clean.shape}")

In [ ]:
print("Mapping columns and standardizing labels...")

# 1. Map Text and Ratings
df_clean['review_text'] = df_clean['Summary'].astype(str)
df_clean['rating'] = pd.to_numeric(df_clean['Rate'], errors='coerce')

# 2. Map Product Details
df_clean['product_title'] = df_clean['product_name'].astype(str)
# Generate a unique dummy product_id based on the product name
df_clean['product_id'] = df_clean['product_title'].astype('category').cat.codes + 1000000

# 3. Standardize Sentiment labels to capitalized ('Positive', 'Neutral', 'Negative')
sentiment_map = {
    'positive': 'Positive',
    'neutral': 'Neutral',
    'negative': 'Negative'
}
df_clean['sentiment'] = df_clean['Sentiment'].str.strip().str.lower().map(sentiment_map)

# Drop any rows where sentiment mapping failed
df_clean = df_clean.dropna(subset=['sentiment', 'rating'])
print(f"Shape after mapping and cleaning: {df_clean.shape}")
print("Sentiment Distribution:")
print(df_clean['sentiment'].value_counts())

In [ ]:
print("Extracting product categories...")

def extract_category(product_name):
    name = str(product_name).lower()
    if any(x in name for x in ["cooler", "fan", "ac", "heater", "purifier", "water purifier", "kettle", "oven"]):
        return "Home & Kitchen"
    elif any(x in name for x in ["lipstick", "shampoo", "serum", "cream", "facial", "makeup", "soap", "hair"]):
        return "Beauty"
    elif any(x in name for x in ["speaker", "alexa", "headphones", "earbuds", "tv", "camera", "mobile", "router", "laptop"]):
        return "Electronics"
    elif any(x in name for x in ["sewing", "tool", "drill", "vacuum", "cleaner", "iron"]):
        return "Tools & Appliances"
    elif any(x in name for x in ["toy", "game", "lego", "board", "puzzle"]):
        return "Toys & Games"
    elif any(x in name for x in ["shirt", "jeans", "scarf", "wool", "dress", "shoes", "bag"]):
        return "Fashion"
    elif any(x in name for x in ["supplement", "multivitamin", "protein", "massager"]):
        return "Health & Personal Care"
    else:
        return "Sports & Outdoors"

df_clean['category'] = df_clean['product_title'].apply(extract_category)
print("Extracted Category Distribution:")
print(df_clean['category'].value_counts())

In [ ]:
print("Performing balanced sampling...")

# Find the minority class size
class_counts = df_clean['sentiment'].value_counts()
min_class_size = class_counts.min()

# We will sample slightly less than the absolute minimum to have a clean round number
SAMPLE_SIZE = int(min(min_class_size, 8800))
print(f"Minority class size: {min_class_size}. Sampling {SAMPLE_SIZE} reviews per class.")

positive_df = df_clean[df_clean['sentiment'] == 'Positive']
neutral_df = df_clean[df_clean['sentiment'] == 'Neutral']
negative_df = df_clean[df_clean['sentiment'] == 'Negative']

positive_sample = positive_df.sample(n=SAMPLE_SIZE, random_state=42)
neutral_sample = neutral_df.sample(n=SAMPLE_SIZE, random_state=42)
negative_sample = negative_df.sample(n=SAMPLE_SIZE, random_state=42)

# Merge and shuffle
balanced_df = pd.concat([positive_sample, neutral_sample, negative_sample])
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Select only the standardized columns required by the ML pipeline
final_columns = ['product_id', 'product_title', 'category', 'review_text', 'rating', 'sentiment']
balanced_df = balanced_df[final_columns]

print(f"\nBalanced Dataset Shape: {balanced_df.shape}")
print("\nBalanced Sentiment Distribution:")
print(balanced_df['sentiment'].value_counts())

print("\nBalanced Category Distribution:")
print(balanced_df['category'].value_counts())

In [ ]:
print(f"Saving balanced dataset to: {BALANCED_PATH}")
balanced_df.to_csv(BALANCED_PATH, index=False)
print("Dataset saved successfully! It is now ready for training.")